In [1]:
# KAGGLE SETUP - Check GPU and install libraries
!nvidia-smi
!pip install tokenizers -q

import os
print("Completed")

Mon Apr 27 06:05:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+
|   1  Tesla T4                       Off |   00000000:00:05.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        

Completed


In [2]:
# KAGGLE SETUP - Mount Google Drive to get our saved data
from google.colab import drive

In [3]:
# KAGGLE SETUP - Install and recreate everything from scratch
!pip install datasets tokenizers lxml cairosvg -q

import os, json, re, numpy as np
from lxml import etree
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer

os.makedirs('/kaggle/working/data', exist_ok=True)
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

def clean_svg(svg_string):
    svg_string = re.sub(r'<!--.*?-->', '', svg_string, flags=re.DOTALL)
    svg_string = re.sub(r'\s+', ' ', svg_string).strip()
    def round_number(match):
        try:
            return str(round(float(match.group()), 1))
        except:
            return match.group()
    svg_string = re.sub(r'-?\d+\.\d+', round_number, svg_string)
    return svg_string

def is_valid_xml(svg_string):
    try:
        etree.fromstring(svg_string.encode())
        return True
    except:
        return False

def is_valid_length(svg_string, min_chars=50, max_chars=8000):
    return min_chars <= len(svg_string) <= max_chars

print("Libraries installed and functions ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/46.0 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.4 MB/s eta 0:00:00


Libraries installed and functions ready


In [4]:
# KAGGLE SETUP - Load and clean all SVG datasets
import random

all_svgs = []

# --- ICONS ---
print("Loading icons...")
ds = load_dataset("starvector/svg-icons-simple")
for item in ds['train']:
    svg = item['Svg']
    if not is_valid_length(svg): continue
    svg = clean_svg(svg)
    if not is_valid_xml(svg): continue
    all_svgs.append(svg)
print(f" Icons: {len(all_svgs):,}")

# --- EMOJI ---
print("Loading emoji...")
ds_emoji = load_dataset("starvector/svg-emoji-simple")
before = len(all_svgs)
for item in ds_emoji['train']:
    svg = item['Svg']
    if not is_valid_length(svg): continue
    svg = clean_svg(svg)
    if not is_valid_xml(svg): continue
    all_svgs.append(svg)
print(f"✅ Emoji added: {len(all_svgs)-before:,}")

# --- FONTS (streaming, 500k) ---
print("Loading fonts (streaming)...")
ds_fonts = load_dataset("starvector/svg-fonts-simple", split="train", streaming=True)
before = len(all_svgs)
for item in ds_fonts:
    svg = item['Svg']
    if not is_valid_length(svg): continue
    svg = clean_svg(svg)
    if not is_valid_xml(svg): continue
    all_svgs.append(svg)
    if len(all_svgs) - before >= 500000: break
    if (len(all_svgs) - before) % 100000 == 0:
        print(f"  Fonts so far: {len(all_svgs)-before:,}")
print(f"Fonts added: {len(all_svgs)-before:,}")

# --- STACK (until 110M tokens) ---
print("Loading stack (streaming)...")
ds_stack = load_dataset("starvector/svg-stack-simple", split="train", streaming=True)
before = len(all_svgs)
for item in ds_stack:
    svg = item['Svg']
    if not is_valid_length(svg): continue
    svg = clean_svg(svg)
    if not is_valid_xml(svg): continue
    all_svgs.append(svg)
    if (len(all_svgs) - before) % 10000 == 0:
        tokens = sum(len(s) for s in all_svgs) / 4
        print(f"  Stack: {len(all_svgs)-before:,} | Total tokens: ~{tokens/1e6:.1f}M")
        if tokens >= 110_000_000: break
print(f"Stack added: {len(all_svgs)-before:,}")

# Shuffle and split
random.seed(42)
random.shuffle(all_svgs)
total = len(all_svgs)
train_end = int(0.98 * total)
val_end   = int(0.99 * total)
train_svgs = all_svgs[:train_end]
val_svgs   = all_svgs[train_end:val_end]
test_svgs  = all_svgs[val_end:]

print(f"\n{'='*40}")
print(f"TOTAL SVGs:  {len(all_svgs):,}")
print(f"Train:       {len(train_svgs):,}")
print(f"Val:         {len(val_svgs):,}")
print(f"Test:        {len(test_svgs):,}")
print(f"Est. tokens: {sum(len(s) for s in all_svgs)/4/1e6:.1f}M")

Loading icons...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/137M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/4.59M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/11.1M [00:00<?, ?B/s]

...

  Fonts so far: 400,000


Fonts added: 500,000
Loading stack (streaming)...


README.md: 0.00B [00:00, ?B/s]

  Stack: 10,000 | Total tokens: ~129.0M
Stack added: 10,000



TOTAL SVGs:  576,776
Train:       565,240
Val:         5,768
Test:        5,768
Est. tokens: 129.0M


In [5]:
# KAGGLE - CELL 3: Train BPE tokenizer and convert to binary
import numpy as np
from tokenizers import ByteLevelBPETokenizer

# Train tokenizer
print("Training BPE tokenizer...")
tokenizer_input = '/kaggle/working/train_text.txt'
with open(tokenizer_input, 'w') as f:
    for svg in train_svgs:
        f.write(svg + '\n')

tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=[tokenizer_input],
    vocab_size=4096,
    min_frequency=2,
    special_tokens=["<pad>", "<s>", "</s>", "<unk>"]
)

os.makedirs('/kaggle/working/tokenizer', exist_ok=True)
tokenizer.save_model('/kaggle/working/tokenizer')
print("✅ Tokenizer trained and saved!")

# Test tokenizer
sample = train_svgs[0]
encoded = tokenizer.encode(sample)
print(f"Sample: {len(sample)} chars → {len(encoded.ids)} tokens")

# Convert to binary
def tokenize_and_save(svgs, split_name):
    print(f"Tokenizing {split_name} ({len(svgs):,} SVGs)...")
    all_tokens = []
    for i, svg in enumerate(svgs):
        tokens = tokenizer.encode(svg).ids
        tokens.append(0)
        all_tokens.extend(tokens)
        if i % 100000 == 0 and i > 0:
            print(f"  {i:,} done...")
    arr = np.array(all_tokens, dtype=np.uint16)
    out_path = f'/kaggle/working/data/{split_name}.bin'
    arr.tofile(out_path)
    print(f"✅ {split_name}: {len(all_tokens):,} tokens saved!")
    return len(all_tokens)

train_count = tokenize_and_save(train_svgs, 'train')
val_count   = tokenize_and_save(val_svgs,   'val')
test_count  = tokenize_and_save(test_svgs,  'test')

print(f"\n{'='*40}")
print(f"Train tokens: {train_count/1e6:.1f}M")
print(f"Val tokens:   {val_count/1e6:.1f}M")
print(f"Test tokens:  {test_count/1e6:.1f}M")
print(f"✅ Binary files ready!")

Training BPE tokenizer...





✅ Tokenizer trained and saved!
Sample: 333 chars → 167 tokens
Tokenizing train (565,240 SVGs)...


  100,000 done...


  200,000 done...


  300,000 done...


  400,000 done...


  500,000 done...


✅ train: 277,106,608 tokens saved!
Tokenizing val (5,768 SVGs)...


✅ val: 2,823,084 tokens saved!
Tokenizing test (5,768 SVGs)...


✅ test: 2,824,405 tokens saved!

Train tokens: 277.1M
Val tokens:   2.8M
Test tokens:  2.8M
✅ Binary files ready!


In [6]:
# KAGGLE - CELL 4: Define SVGTransformer model class
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config['n_head']
        self.n_embd = config['n_embd']
        self.c_attn = nn.Linear(config['n_embd'], 3 * config['n_embd'])
        self.c_proj = nn.Linear(config['n_embd'], config['n_embd'])
        self.register_buffer('bias', torch.tril(
            torch.ones(config['block_size'], config['block_size']))
            .view(1,1,config['block_size'],config['block_size']))

    def forward(self, x):
        B,T,C = x.size()
        q,k,v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        q = q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v = v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        att = (q @ k.transpose(-2,-1)) * (1.0/np.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T]==0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.c_proj(y)

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config['n_embd'], 4*config['n_embd'])
        self.c_proj = nn.Linear(4*config['n_embd'], config['n_embd'])
        self.act    = nn.GELU()
    def forward(self, x):
        return self.c_proj(self.act(self.c_fc(x)))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config['n_embd'])
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config['n_embd'])
        self.mlp  = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class SVGTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config['vocab_size'], config['n_embd']),
            wpe = nn.Embedding(config['block_size'], config['n_embd']),
            h   = nn.ModuleList([Block(config) for _ in range(config['n_layer'])]),
            ln_f= nn.LayerNorm(config['n_embd']),
        ))
        self.lm_head = nn.Linear(config['n_embd'], config['vocab_size'], bias=False)

    def forward(self, idx, targets=None):
        B,T = idx.size()
        pos = torch.arange(0, T, device=idx.device)
        x = self.transformer.wte(idx) + self.transformer.wpe(pos)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

print("SVGTransformer model class defined!")

# Verify GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# Quick test - create tiny model and check params
test_config = {
    'vocab_size': 4096, 'block_size': 256,
    'batch_size': 32, 'n_layer': 4,
    'n_head': 4, 'n_embd': 128, 'lr': 0.001
}
test_model = SVGTransformer(test_config).to(device)
print(f"Test model params: {test_model.count_params()/1e6:.2f}M")
print(" Model works on", device)

SVGTransformer model class defined!
Device: cuda


Test model params: 1.87M
 Model works on cuda


In [7]:
# Quick check - verify binary files exist
import os
import numpy as np

files = ['train.bin', 'val.bin', 'test.bin']
for f in files:
    path = f'/kaggle/working/data/{f}'
    if os.path.exists(path):
        data = np.memmap(path, dtype=np.uint16, mode='r')
        print(f"{f}: {len(data):,} tokens")
    else:
        print(f"{f}: NOT FOUND - need to rerun Cell 3")
        

train.bin: 277,106,608 tokens
val.bin: 2,823,084 tokens
test.bin: 2,824,405 tokens


In [8]:
# KAGGLE - CELL 5: Define training utilities
import numpy as np
import torch
import os

def get_batch(split, block_size, batch_size, device, data_dir='/kaggle/working/data'):
    if split == 'train':
        data = np.memmap(os.path.join(data_dir, 'train.bin'),
                        dtype=np.uint16, mode='r')
    else:
        data = np.memmap(os.path.join(data_dir, 'val.bin'),
                        dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(
            (data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(
            (data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

def get_lr(it, max_lr, warmup_iters, max_iters):
    # Linear warmup
    if it < warmup_iters:
        return max_lr * it / warmup_iters
    # Cosine decay
    t = (it - warmup_iters) / (max_iters - warmup_iters)
    return max_lr * 0.5 * (1.0 + np.cos(np.pi * t))

# Test get_batch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
x, y = get_batch('train', 256, 32, device)
print(f" get_batch works!")
print(f"  x shape: {x.shape}  (batch_size × block_size)")
print(f"  y shape: {y.shape}  (same, shifted by 1)")
print(f"  device:  {x.device}")

 get_batch works!
  x shape: torch.Size([32, 256])  (batch_size × block_size)
  y shape: torch.Size([32, 256])  (same, shifted by 1)
  device:  cuda:0


In [9]:
# KAGGLE - CELL 6: Learning rate sweep on Tiny model
import time

def train_model(config, max_iters=500):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = SVGTransformer(config).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config['lr'],
        betas=(0.9, 0.95), weight_decay=0.1)

    for it in range(max_iters):
        lr = get_lr(it, config['lr'], 50, max_iters)
        for g in optimizer.param_groups:
            g['lr'] = lr
        x, y = get_batch('train', config['block_size'],
                         config['batch_size'], device)
        _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

    # Validation loss
    model.eval()
    val_losses = []
    with torch.no_grad():
        for _ in range(50):
            x, y = get_batch('val', config['block_size'],
                             config['batch_size'], device)
            _, loss = model(x, y)
            val_losses.append(loss.item())
    return np.mean(val_losses), model.count_params()

# Tiny base config
tiny_base = {
    'vocab_size': 4096,
    'block_size': 256,
    'batch_size': 32,
    'n_layer': 4,
    'n_head': 4,
    'n_embd': 128,
}

# Sweep
learning_rates = [1e-4, 3e-4, 1e-3, 3e-3, 6e-3, 1e-2, 3e-2]
sweep_results = []

print(" LR Sweep on Tiny Model (500 iters each)...")
print(f"{'LR':>10} | {'Val Loss':>10} | {'Time':>8}")
print("-" * 35)

for lr in learning_rates:
    config = {**tiny_base, 'lr': lr}
    t0 = time.time()
    val_loss, n_params = train_model(config, max_iters=500)
    elapsed = time.time() - t0
    sweep_results.append({'lr': lr, 'val_loss': val_loss})
    print(f"{lr:>10.4f} | {val_loss:>10.4f} | {elapsed:>6.0f}s")

best = min(sweep_results, key=lambda x: x['val_loss'])
best_lr = best['lr']
print(f"\n Best LR: {best_lr} (val_loss={best['val_loss']:.4f})")

 LR Sweep on Tiny Model (500 iters each)...
        LR |   Val Loss |     Time
-----------------------------------


    0.0001 |     2.5030 |     31s


    0.0003 |     1.6764 |     23s


    0.0010 |     1.3295 |     23s


    0.0030 |     1.1614 |     24s


    0.0060 |     1.0533 |     25s


    0.0100 |     0.9699 |     26s


    0.0300 |     0.9874 |     26s

 Best LR: 0.01 (val_loss=0.9699)


In [10]:
# KAGGLE - CELL 7: Train Tiny, Small, Medium (SKIPS IF ALREADY DONE)
import time, os, json
import numpy as np
import torch

best_lr = 0.01
total_train_tokens = 277_106_608
batch_size = 32
block_size = 256
max_iters = total_train_tokens // (batch_size * block_size)

model_configs_small = {
    'tiny':   {'n_layer':4,  'n_head':4,  'n_embd':128},
    'small':  {'n_layer':6,  'n_head':6,  'n_embd':192},
    'medium': {'n_layer':6,  'n_head':6,  'n_embd':384},
}

def train_full(name, arch, lr, max_iters):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    config = {
        'vocab_size': 4096,
        'block_size': block_size,
        'batch_size': batch_size,
        'lr': lr,
        **arch
    }
    model = SVGTransformer(config).to(device)
    n_params = model.count_params()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr,
        betas=(0.9, 0.95), weight_decay=0.1)

    train_losses, val_losses, log_iters = [], [], []

    print(f"\n{'='*55}")
    print(f"Training {name.upper()} | {n_params/1e6:.2f}M params | "
          f"LR={lr} | {max_iters:,} iters")
    print(f"{'='*55}")
    t_start = time.time()

    for it in range(max_iters):
        lr_it = get_lr(it, lr, 200, max_iters)
        for g in optimizer.param_groups:
            g['lr'] = lr_it

        model.train()
        x, y = get_batch('train', block_size, batch_size, device)
        _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

        if it % 500 == 0 or it == max_iters - 1:
            model.eval()
            v_losses = []
            with torch.no_grad():
                for _ in range(20):
                    x, y = get_batch('val', block_size, batch_size, device)
                    _, vloss = model(x, y)
                    v_losses.append(vloss.item())
            val_loss = np.mean(v_losses)
            train_losses.append(loss.item())
            val_losses.append(val_loss)
            log_iters.append(it)
            elapsed = time.time() - t_start
            eta = (elapsed / (it+1)) * (max_iters - it - 1)
            print(f"  iter {it:5d}/{max_iters} | "
                  f"train={loss.item():.4f} | "
                  f"val={val_loss:.4f} | "
                  f"elapsed={elapsed/60:.1f}m | "
                  f"eta={eta/60:.1f}m")

    os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
    torch.save({
        'model_state': model.state_dict(),
        'config': config,
        'val_loss': val_losses[-1],
        'n_params': n_params,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'log_iters': log_iters,
    }, f'/kaggle/working/checkpoints/{name}.pt')
    print(f"✅ {name} saved! Final val_loss={val_losses[-1]:.4f}")

    return {
        'name': name,
        'n_params': n_params,
        'final_val_loss': val_losses[-1],
        'train_losses': train_losses,
        'val_losses': val_losses,
        'log_iters': log_iters,
        'wall_time': time.time() - t_start,
    }

# ── SKIP IF ALREADY TRAINED ──
already_done = all([
    os.path.exists(f'/kaggle/working/checkpoints/{n}.pt')
    for n in ['tiny', 'small', 'medium']
])

if already_done:
    print("✅ Tiny, Small, Medium already trained — skipping!")
    with open('/kaggle/working/tiny_small_medium_results.json') as f:
        all_results = json.load(f)
    print(f"\n{'Model':>8} | {'Params':>10} | {'Val Loss':>10}")
    print("-" * 35)
    for name, r in all_results.items():
        print(f"{name:>8} | {r['n_params']/1e6:>8.2f}M | "
              f"{r['final_val_loss']:>10.4f}")
else:
    print("Training Tiny, Small, Medium from scratch...")
    all_results = {}
    for name, arch in model_configs_small.items():
        result = train_full(name, arch, best_lr, max_iters)
        all_results[name] = result

    # Save results
    with open('/kaggle/working/tiny_small_medium_results.json', 'w') as f:
        json.dump({
            name: {
                'n_params': r['n_params'],
                'final_val_loss': r['final_val_loss'],
                'wall_time': r['wall_time'],
                'train_losses': r['train_losses'],
                'val_losses': r['val_losses'],
                'log_iters': r['log_iters'],
            } for name, r in all_results.items()
        }, f)

    print(f"\n{'Model':>8} | {'Params':>10} | {'Val Loss':>10}")
    print("-" * 35)
    for name, r in all_results.items():
        print(f"{name:>8} | {r['n_params']/1e6:>8.2f}M | "
              f"{r['final_val_loss']:>10.4f}")

Training Tiny, Small, Medium from scratch...

Training TINY | 1.87M params | LR=0.01 | 33,826 iters


  iter     0/33826 | train=8.4642 | val=8.4524 | elapsed=0.0m | eta=235.6m


  iter   500/33826 | train=0.9182 | val=0.9416 | elapsed=0.4m | eta=28.0m


  iter  1000/33826 | train=0.8734 | val=0.8677 | elapsed=0.8m | eta=27.6m


  iter  1500/33826 | train=0.8253 | val=0.8311 | elapsed=1.3m | eta=27.2m


...

  iter 32000/33826 | train=0.5978 | val=0.6205 | elapsed=138.3m | eta=7.9m


  iter 32500/33826 | train=0.6039 | val=0.6112 | elapsed=140.5m | eta=5.7m


  iter 33000/33826 | train=0.6164 | val=0.6161 | elapsed=142.7m | eta=3.6m


  iter 33500/33826 | train=0.6080 | val=0.6073 | elapsed=144.9m | eta=1.4m


  iter 33825/33826 | train=0.6172 | val=0.6102 | elapsed=146.3m | eta=0.0m
✅ medium saved! Final val_loss=0.6102

   Model |     Params |   Val Loss
-----------------------------------
    tiny |     1.87M |     0.5878
   small |     4.29M |     0.5788
  medium |    13.89M |     0.6102


In [11]:
# Save tiny/small/medium results
import json

with open('/kaggle/working/tiny_small_medium_results.json', 'w') as f:
    json.dump({
        name: {
            'n_params': r['n_params'],
            'final_val_loss': r['final_val_loss'],
            'wall_time': r['wall_time'],
            'train_losses': r['train_losses'],
            'val_losses': r['val_losses'],
            'log_iters': r['log_iters'],
        } for name, r in all_results.items()
    }, f)
print("Results saved!")
print("tiny:   val_loss=0.6344")
print("small:  val_loss=0.6000")
print("medium: val_loss=0.6366")

Results saved!
tiny:   val_loss=0.6344
small:  val_loss=0.6000
medium: val_loss=0.6366


In [12]:
# KAGGLE - CELL 7B: Train Large and XL using 2x T4 GPUs
import time, os
import numpy as np
import torch
import torch.nn as nn

print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

best_lr = 0.01
total_train_tokens = 277_106_608
batch_size = 64  # doubled for 2 GPUs
block_size = 256
max_iters = total_train_tokens // (batch_size * block_size)
print(f"1 epoch = {max_iters:,} iterations")

model_configs_large = {
    'large': {'n_layer':10, 'n_head':8,  'n_embd':512},
    'xl':    {'n_layer':12, 'n_head':12, 'n_embd':768},
}

def train_full_multigpu(name, arch, lr, max_iters):
    device = 'cuda'
    config = {
        'vocab_size': 4096,
        'block_size': block_size,
        'batch_size': batch_size,
        'lr': lr,
        **arch
    }
    model = SVGTransformer(config).to(device)
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)

    n_params = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr,
        betas=(0.9, 0.95), weight_decay=0.1)

    train_losses, val_losses, log_iters = [], [], []

    print(f"\n{'='*55}")
    print(f"Training {name.upper()} | {n_params/1e6:.2f}M params | "
          f"LR={lr} | {max_iters:,} iters")
    print(f"{'='*55}")
    t_start = time.time()

    for it in range(max_iters):
        lr_it = get_lr(it, lr, 200, max_iters)
        for g in optimizer.param_groups:
            g['lr'] = lr_it

        model.train()
        x, y = get_batch('train', block_size, batch_size, device)
        _, loss = model(x, y)
        loss = loss.mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

        if it % 500 == 0 or it == max_iters - 1:
            model.eval()
            v_losses = []
            with torch.no_grad():
                for _ in range(20):
                    x, y = get_batch('val', block_size, batch_size, device)
                    _, vloss = model(x, y)
                    v_losses.append(vloss.mean().item())
            val_loss = np.mean(v_losses)
            train_losses.append(loss.item())
            val_losses.append(val_loss)
            log_iters.append(it)
            elapsed = time.time() - t_start
            eta = (elapsed / (it+1)) * (max_iters - it - 1)
            print(f"  iter {it:5d}/{max_iters} | "
                  f"train={loss.item():.4f} | "
                  f"val={val_loss:.4f} | "
                  f"elapsed={elapsed/60:.1f}m | "
                  f"eta={eta/60:.1f}m")

    os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
    model_to_save = model.module if hasattr(model, 'module') else model
    torch.save({
        'model_state': model_to_save.state_dict(),
        'config': config,
        'val_loss': val_losses[-1],
        'n_params': n_params,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'log_iters': log_iters,
    }, f'/kaggle/working/checkpoints/{name}.pt')
    print(f" {name} saved! Final val_loss={val_losses[-1]:.4f}")

    return {
        'name': name,
        'n_params': n_params,
        'final_val_loss': val_losses[-1],
        'train_losses': train_losses,
        'val_losses': val_losses,
        'log_iters': log_iters,
        'wall_time': time.time() - t_start,
    }

all_results_2 = {}
for name, arch in model_configs_large.items():
    result = train_full_multigpu(name, arch, best_lr, max_iters)
    all_results_2[name] = result

print(f"\n{'Model':>8} | {'Params':>10} | {'Val Loss':>10} | {'Time':>10}")
print("-" * 50)
for name, r in all_results_2.items():
    print(f"{name:>8} | {r['n_params']/1e6:>8.2f}M | "
          f"{r['final_val_loss']:>10.4f} | "
          f"{r['wall_time']/60:>8.1f}m")

GPUs available: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
1 epoch = 16,913 iterations


Using 2 GPUs!

Training LARGE | 35.85M params | LR=0.01 | 16,913 iters


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


  iter     0/16913 | train=8.4602 | val=8.4624 | elapsed=0.1m | eta=1892.9m


  iter   500/16913 | train=2.0870 | val=2.0960 | elapsed=5.8m | eta=191.2m


...

  iter  5000/16913 | train=1.6712 | val=1.6115 | elapsed=140.5m | eta=334.6m


  iter  5500/16913 | train=1.4016 | val=1.3966 | elapsed=154.5m | eta=320.6m


  iter  6000/16913 | train=1.1304 | val=1.1501 | elapsed=168.6m | eta=306.6m


  iter  6500/16913 | train=0.9730 | val=0.9671 | elapsed=182.7m | eta=292.6m


  iter  7000/16913 | train=0.8634 | val=0.8717 | elapsed=196.9m | eta=278.8m


In [ ]:
# KAGGLE - FINAL CELL: Combine and save all 5 results
import json

# Load tiny/small/medium
with open('/kaggle/working/tiny_small_medium_results.json') as f:
    all_results = json.load(f)

# Add large/xl
for name, r in all_results_2.items():
    all_results[name] = {
        'n_params': r['n_params'],
        'final_val_loss': r['final_val_loss'],
        'wall_time': r['wall_time'],
        'train_losses': r['train_losses'],
        'val_losses': r['val_losses'],
        'log_iters': r['log_iters'],
    }

# Save combined
with open('/kaggle/working/all_5_results.json', 'w') as f:
    json.dump(all_results, f)

print("ALL 5 MODELS COMPLETED")
print(f"\n{'Model':>8} | {'Params':>10} | {'Val Loss':>10}")
print("-" * 35)
for name, r in all_results.items():
    print(f"{name:>8} | {r['n_params']/1e6:>8.2f}M | "
          f"{r['final_val_loss']:>10.4f}")